# Production Deployment with NAT UI

In [1]:
%%capture
# load env variables
import os
from dotenv import load_dotenv
load_dotenv()

# Install the climate analyzer package
!cd climate_analyzer && pip install -e . && cd ..

## Start the NAT Server

In [2]:
# start the NAT server
import subprocess
import time

# Start NAT with explicit IPv4 host
nat_process = subprocess.Popen(
    ["nat", "serve", "--config_file", "./climate_analyzer/src/climate_analyzer/configs/config.yml", 
     "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for it to start
time.sleep(10)

# Check if it crashed
if nat_process.poll() is not None:
    stdout, stderr = nat_process.communicate()
    print("❌ Server crashed!")
    print("\n=== Error Output ===")
    print(stderr[-500:])
else:
    print("✅ Server is running in the background!")

✅ Server is running in the background!


## Install NeMo Agent Toolkit UI and Dependencies

- **Clone the NAT UI Repository** - Clones the NAT UI repository from GitHub if it doesn't already exist locally
- **Installs JavaScript dependencies** – Uses `npm install` to fetch React, Next.js, and other required packages, creating the node_modules/ folder with ~1000 packages. We've already done this for you on the deeplearning platform, but if you are running on your own environment, make sure to uncomment and run this block of code. 

**This step may take several minutes.**

Without these steps, the UI won’t start because it’s missing dependencies and the built assets (similar to trying to run Python code without installing required packages via pip).

In [3]:
from pathlib import Path

ui_path = Path("NeMo-Agent-Toolkit-UI")

if not ui_path.exists():
    print("Cloning NAT UI repository...")
    subprocess.run([
        "git", "clone",
        "https://github.com/NVIDIA/NeMo-Agent-Toolkit-UI.git"
    ], check=True)
    print("UI repository cloned")
else:
    print("UI respository already exists!")

UI respository already exists!


## Start UI

This code starts the NeMo Agent Toolkit web UI in **development mode** in the background, so you can interact with your agent through a browser.

In [5]:
# start UI
print("Starting UI development server...")
ui_process = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=ui_path,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env={**os.environ, "NEXT_TELEMETRY_DISABLED": "1"}
)

time.sleep(30)
print(f"UI started!")

Starting UI development server...
UI started!


## Access the UI

In [6]:
import os

def make_nat_ui_url(port=3000):
    """Generate the accessible URL for NAT UI on DeepLearning.AI platform."""
    
    # Extract IP from hostname (same pattern as Phoenix)
    ip = os.environ['HOSTNAME'].split('.')[0][3:]
    
    # Build the URL using platform's reverse proxy pattern
    url = os.environ['REV_PROXY_BASE_DOMAIN'].format(ip=ip, port=port)
    
    # Terminal formatting
    BOLD = "\033[1m"
    RESET = "\033[0m"
    
    print(f"{BOLD}NAT UI URL: {url}{RESET}")
    return url

# Try both ports (one should work)
print("Your agent can be accessed at the following URL:\n")
url_proxy = make_nat_ui_url(3000)  # Proxy server


Your agent can be accessed at the following URL:

NAT UI URL: https://s172-29-90-130p3000.lab-aws-production.deeplearning.ai


In [7]:
# ===== Cleanup =====
print("🛑 Stopping services...")
nat_process.terminate()
nat_process.wait()
!pkill -f "npm run dev" 2>/dev/null || true
print("✅ Services stopped")

🛑 Stopping services...
✅ Services stopped
